In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Verify your folders are actually there**

In [ ]:
import os
base = "/content/drive/MyDrive/Fine Tuning Whisper Model"
for f in os.listdir(base):
    print(f)

Thesis Cleaned Audio
Thesis Whisper Labels


**Install everything needed**

In [ ]:
!pip install -q pydub transformers datasets accelerate evaluate jiwer librosa soundfile

**Build the dataset**

In [ ]:
import os, csv
from pydub import AudioSegment

BASE = "/content/drive/MyDrive/Fine Tuning Whisper Model"
CLEANED_AUDIO_DIR = BASE + "/Thesis Cleaned Audio"
LABELS_DIR = BASE + "/Thesis Whisper Labels"
OUTPUT_CLIPS_DIR = BASE + "/clean_clips"
METADATA_PATH = BASE + "/metadata.csv"

os.makedirs(OUTPUT_CLIPS_DIR, exist_ok=True)

audio_lookup = {}
for root, dirs, files in os.walk(CLEANED_AUDIO_DIR):
    for f in files:
        if f.lower().endswith(".wav"):
            audio_lookup[os.path.splitext(f)[0]] = os.path.join(root, f)

print(f"Found {len(audio_lookup)} cleaned audio files.")

rows = []
clip_counter = 1
missing = []

for root, dirs, files in os.walk(LABELS_DIR):
    for label_file in files:
        if not label_file.lower().endswith(".txt"):
            continue
        base_name = os.path.splitext(label_file)[0]

        if base_name not in audio_lookup:
            missing.append(base_name)
            continue

        audio = AudioSegment.from_wav(audio_lookup[base_name]).set_channels(1).set_frame_rate(16000)

        with open(os.path.join(root, label_file), encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) < 3:
                    continue
                try:
                    start_sec, end_sec = float(parts[0]), float(parts[1])
                except ValueError:
                    continue
                text = parts[2]
                if end_sec <= start_sec:
                    continue

                clip = audio[int(start_sec * 1000):int(end_sec * 1000)]
                clip_name = f"clip_{clip_counter:04d}.wav"
                clip.export(os.path.join(OUTPUT_CLIPS_DIR, clip_name), format="wav")

                rows.append({
                    "audio_file": clip_name,
                    "text": text.strip(),
                    "duration_sec": round(end_sec - start_sec, 2),
                    "source_recording": base_name,
                })
                clip_counter += 1

with open(METADATA_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["audio_file", "text", "duration_sec", "source_recording"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone: {clip_counter - 1} clips created from {len(set(r['source_recording'] for r in rows))} source files.")
if missing:
    print(f"{len(missing)} labels had no matching audio - skipped:")
    for m in missing[:20]:
        print(" -", m)

Found 95 cleaned audio files.

Done: 2179 clips created from 93 source files.
1 labels had no matching audio - skipped:
 - Harrapa_Noor_Kharal_1


**Split into train / validation / test**

In [ ]:
import pandas as pd

df = pd.read_csv(METADATA_PATH).sample(frac=1, random_state=42).reset_index(drop=True)
n = len(df)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

df[:train_end].to_csv(f"{BASE}/train.csv", index=False)
df[train_end:val_end].to_csv(f"{BASE}/val.csv", index=False)
df[val_end:].to_csv(f"{BASE}/test.csv", index=False)

print(f"train: {train_end} | val: {val_end-train_end} | test: {n-val_end}")

train: 1743 | val: 218 | test: 218


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

**Load the processor**

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="urdu", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="urdu", task="transcribe")

**Load dataset and prepare audio/text features**

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict, Audio

CLIPS_DIR = f"{BASE}/clean_clips"

def load_split(csv_path):
    d = pd.read_csv(csv_path)
    d["audio"] = d["audio_file"].apply(lambda x: f"{CLIPS_DIR}/{x}")
    ds = Dataset.from_pandas(d[["audio", "text"]])
    return ds.cast_column("audio", Audio(sampling_rate=16000))

dataset = DatasetDict({
    "train": load_split(f"{BASE}/train.csv"),
    "validation": load_split(f"{BASE}/val.csv"),
    "test": load_split(f"{BASE}/test.csv"),
})
print(dataset)

NameError: name 'BASE' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q pydub transformers datasets accelerate evaluate jiwer librosa soundfile

import pandas as pd
from datasets import Dataset, DatasetDict, Audio

BASE = "/content/drive/MyDrive/Fine Tuning Whisper Model"
CLIPS_DIR = f"{BASE}/clean_clips"

def load_split(csv_path):
    d = pd.read_csv(csv_path)
    d["audio"] = d["audio_file"].apply(lambda x: f"{CLIPS_DIR}/{x}")
    ds = Dataset.from_pandas(d[["audio", "text"]])
    return ds.cast_column("audio", Audio(sampling_rate=16000))

dataset = DatasetDict({
    "train": load_split(f"{BASE}/train.csv"),
    "validation": load_split(f"{BASE}/val.csv"),
    "test": load_split(f"{BASE}/test.csv"),
})
print("BASE =", BASE)
print(dataset)

Mounted at /content/drive
BASE = /content/drive/MyDrive/Fine Tuning Whisper Model
DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 1743
    })
    validation: Dataset({
        features: ['audio', 'text'],
        num_rows: 218
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 218
    })
})


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="urdu", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="urdu", task="transcribe")

In [ ]:
def prepare_dataset_manual(hf_dataset, split_name):
    input_features_list = []
    labels_list = []
    total = len(hf_dataset)
    for i, example in enumerate(hf_dataset):
        audio = example["audio"]
        input_features_list.append(
            feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
        )
        labels_list.append(tokenizer(example["text"]).input_ids)
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"{split_name}: {i + 1}/{total}")
    return Dataset.from_dict({"input_features": input_features_list, "labels": labels_list})

train_processed = prepare_dataset_manual(dataset["train"], "train")
val_processed = prepare_dataset_manual(dataset["validation"], "validation")
test_processed = prepare_dataset_manual(dataset["test"], "test")

dataset = DatasetDict({"train": train_processed, "validation": val_processed, "test": test_processed})
print(dataset)

train: 100/1743
train: 200/1743
train: 300/1743
train: 400/1743
train: 500/1743
train: 600/1743
train: 700/1743
train: 800/1743
train: 900/1743
train: 1000/1743
train: 1100/1743
train: 1200/1743
train: 1300/1743
train: 1400/1743
train: 1500/1743
train: 1600/1743
train: 1700/1743
train: 1743/1743


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q pydub transformers datasets accelerate evaluate jiwer librosa soundfile

import pandas as pd
from datasets import Dataset, DatasetDict, Audio

BASE = "/content/drive/MyDrive/Fine Tuning Whisper Model"
CLIPS_DIR = f"{BASE}/clean_clips"

def load_split(csv_path):
    d = pd.read_csv(csv_path)
    d["audio"] = d["audio_file"].apply(lambda x: f"{CLIPS_DIR}/{x}")
    ds = Dataset.from_pandas(d[["audio", "text"]])
    return ds.cast_column("audio", Audio(sampling_rate=16000))

dataset = DatasetDict({
    "train": load_split(f"{BASE}/train.csv"),
    "validation": load_split(f"{BASE}/val.csv"),
    "test": load_split(f"{BASE}/test.csv"),
})
print("BASE =", BASE)
print(dataset)

Mounted at /content/drive
BASE = /content/drive/MyDrive/Fine Tuning Whisper Model
DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 1743
    })
    validation: Dataset({
        features: ['audio', 'text'],
        num_rows: 218
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 218
    })
})


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="urdu", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="urdu", task="transcribe")

In [ ]:
from datasets import Dataset, DatasetDict

def make_generator(hf_dataset):
    def gen():
        for example in hf_dataset:
            audio = example["audio"]
            input_features = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
            labels = tokenizer(example["text"]).input_ids
            yield {"input_features": input_features, "labels": labels}
    return gen

train_processed = Dataset.from_generator(make_generator(dataset["train"]))
val_processed = Dataset.from_generator(make_generator(dataset["validation"]))
test_processed = Dataset.from_generator(make_generator(dataset["test"]))

dataset = DatasetDict({"train": train_processed, "validation": val_processed, "test": test_processed})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1743
    })
    validation: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 218
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 218
    })
})


In [ ]:
dataset.save_to_disk(f"{BASE}/processed_dataset")
print("Checkpoint saved.")

Checkpoint saved.


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.generation_config.language = "urdu"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
import torch, evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=f"{BASE}/whisper-jangli-small",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=4,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=25,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    report_to=["none"],
)

In [14]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
trainer.train()

Step,Training Loss,Validation Loss,Wer
200,1.597692,0.923879,85.516467


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Wer
200,1.597692,0.923879,85.516467
400,0.836388,0.956137,82.597305
436,0.797900,0.958037,82.522455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=436, training_loss=1.7396368411702847, metrics={'train_runtime': 2259.1237, 'train_samples_per_second': 3.086, 'train_steps_per_second': 0.193, 'total_flos': 2.01201741103104e+18, 'train_loss': 1.7396368411702847, 'epoch': 4.0})

In [15]:
trainer.save_model(f"{BASE}/whisper-jangli-small-final")
processor.save_pretrained(f"{BASE}/whisper-jangli-small-final")
print("Model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


**Baseline Vs Fine Tuned**

In [16]:
baseline_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
baseline_model.generation_config.language = "urdu"
baseline_model.generation_config.task = "transcribe"
baseline_model.generation_config.forced_decoder_ids = None

def evaluate_model(eval_model, test_dataset):
    predictions, references = [], []
    eval_model.eval()
    for example in test_dataset:
        input_features = torch.tensor(example["input_features"]).unsqueeze(0).to("cuda")
        with torch.no_grad():
            predicted_ids = eval_model.generate(input_features, max_length=225)
        pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        label_ids = [l if l != -100 else tokenizer.pad_token_id for l in example["labels"]]
        ref_text = tokenizer.decode(label_ids, skip_special_tokens=True)
        predictions.append(pred_text)
        references.append(ref_text)
    wer_score = 100 * metric.compute(predictions=predictions, references=references)
    return wer_score, predictions, references

baseline_wer, _, _ = evaluate_model(baseline_model, dataset["test"])
print(f"BASELINE WER: {baseline_wer:.2f}%")

model.to("cuda")
finetuned_wer, fine_predictions, fine_references = evaluate_model(model, dataset["test"])
print(f"FINE-TUNED WER: {finetuned_wer:.2f}%")
print(f"Improvement: {baseline_wer - finetuned_wer:.2f} points")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

BASELINE WER: 108.66%
FINE-TUNED WER: 86.09%
Improvement: 22.57 points


***Save Results***

In [17]:
results_df = pd.DataFrame({"actual_transcript": fine_references[:10], "model_prediction": fine_predictions[:10]})
results_df.to_csv(f"{BASE}/sample_predictions.csv", index=False)

with open(f"{BASE}/results_summary.txt", "w") as f:
    f.write(f"Baseline WER: {baseline_wer:.2f}%\n")
    f.write(f"Fine-tuned WER: {finetuned_wer:.2f}%\n")
    f.write(f"Improvement: {baseline_wer - finetuned_wer:.2f} points\n")
    f.write(f"Training clips: {len(dataset['train'])}\n")

print("Saved.")

Saved.


***Prepare the sample metadata file***

In [18]:
import pandas as pd
df = pd.read_csv(f"{BASE}/metadata.csv")
df.head(30).to_csv(f"{BASE}/metadata_sample.csv", index=False)
print("Sample saved.")

Sample saved.
